# Re-MFAR

In [8]:
import nest_asyncio

nest_asyncio.apply()

In [9]:
import torch
from torch.utils import data

### Term Indexes

#### Dummy docs and vocabs using trie and csc_matrix

In [10]:
# Python: build a (16,100) CSC matrix and a marisa_trie of 100 one-word animal names

import numpy as np
from scipy.sparse import csc_matrix

# 100 one-word English animal names (same as before)
animals = [
    "dog","cat","horse","cow","sheep","goat","pig","rabbit","mouse","rat",
    "hamster","gerbil","bat","fox","wolf","lion","tiger","bear","elephant","rhino",
    "hippo","giraffe","zebra","moose","deer","elk","bison","buffalo","kangaroo","koala",
    "wombat","platypus","otter","beaver","raccoon","skunk","seal","whale","dolphin","porpoise",
    "shark","eel","octopus","squid","crab","lobster","shrimp","jellyfish","starfish","seahorse",
    "penguin","albatross","sparrow","pigeon","dove","crow","raven","owl","hawk","eagle",
    "falcon","kestrel","parrot","macaw","canary","finch","pelican","flamingo","swan","goose",
    "duck","turkey","chicken","rooster","hen","camel","llama","alpaca","yak","marmoset",
    "lemur","monkey","chimpanzee","gorilla","orangutan","iguana","lizard","snake","cobra","viper",
    "turtle","tortoise","frog","toad","salamander","newt","centipede","millipede","bee","wasp"
]
assert len(animals) == 100

# build marisa trie & token2id
import marisa_trie
vocab = sorted(animals)
trie = marisa_trie.Trie(vocab)

# build 16 documents, each doc length between 20 and 30 tokens (sampled with replacement)
rng = np.random.default_rng(42)
n_docs = 16
docs = []
for i in range(n_docs):
    L = int(rng.integers(20, 31))  # 20..30 tokens
    tokens = list(rng.choice(vocab, size=L, replace=True))
    docs.append({"title": f"doc_{i}", "text": " ".join(tokens)})

# build term-document matrix (counts) shape (n_docs, n_vocab)
n_vocab = len(vocab)
dense = np.zeros((n_docs, n_vocab), dtype=np.float32)
for d_idx, doc in enumerate(docs):
    for tok in doc["text"].split():
        dense[d_idx, trie[tok]] += 1.0

term_index = csc_matrix(dense, dtype=np.float32)

# quick checks
print("n_docs:", n_docs)
print("n_vocab:", n_vocab)
print("term_index.shape:", term_index.shape)
print("nnz:", term_index.nnz)
print("example doc[0] length tokens:", int(dense[0].sum()))
print("example doc[0] sample tokens:", docs[0]["text"].split()[:10])


n_docs: 16
n_vocab: 100
term_index.shape: (16, 100)
nnz: 353
example doc[0] length tokens: 20
example doc[0] sample tokens: ['seahorse', 'penguin', 'jellyfish', 'jellyfish', 'squid', 'camel', 'porpoise', 'dolphin', 'canary', 'macaw']


In [11]:
term_index[15, trie["dog"]]

1.0

In [12]:
def get_term_score(token: str, doc_id: int) -> float:
    """Get the term score (count) for a given token in a given document."""
    if token not in trie:
        return 0.0
    token_id: int = trie[token]
    return term_index[doc_id, token_id]

### Datasets

In [13]:
# Dataset class for bi-encoder training
class BiEncoderDataset(data.Dataset):

    def __init__(self,
                 tokenizer,
                 data,
                 max_length=512,
                 item_keys={
                     'query': 'question',
                     'passage': 'section_text'
                 }):
        """
        Args:
            tokenizer: HuggingFace tokenizer.
            data: List of dictionaries with 'query' and 'passage'.
            max_length: Max tokenized sequence length.
        """
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.query_key = item_keys['query']
        self.passage_key = item_keys['passage']

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        query = item[self.query_key]
        passage = item[self.passage_key]

        # Tokenize query and passage
        query_encoding = self.tokenizer(query,
                                        padding='max_length',
                                        truncation=True,
                                        max_length=self.max_length,
                                        return_tensors="pt")
        passage_encoding = self.tokenizer(passage,
                                          padding='max_length',
                                          truncation=True,
                                          max_length=self.max_length,
                                          return_tensors="pt")

        # Flatten tensors and return
        return {
            'input_ids_query': query_encoding['input_ids'].squeeze(0),
            'attention_mask_query':
            query_encoding['attention_mask'].squeeze(0),
            'input_ids_passage': passage_encoding['input_ids'].squeeze(0),
            'attention_mask_passage':
            passage_encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(1)  # For contrastive training, dummy label
        }

### Encoder Models

In [14]:
# Define a bi-encoder model using a shared transformer
class BiEncoderModel(torch.nn.Module):
    def __init__(self, model_name):
        """
        Args:
            model_name: Name of a pretrained transformer model.
        """
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)

    def forward(self, input_ids_query, attention_mask_query, input_ids_passage, attention_mask_passage):
        # Embed queries
        query_outputs = self.encoder(
            input_ids=input_ids_query, attention_mask=attention_mask_query
        )
        query_embeddings = query_outputs.last_hidden_state[:, 0, :]  # CLS token embedding

        # Embed passages
        passage_outputs = self.encoder(
            input_ids=input_ids_passage, attention_mask=attention_mask_passage
        )
        passage_embeddings = passage_outputs.last_hidden_state[:, 0, :]  # CLS token embedding

        return query_embeddings, passage_embeddings



### Losses

In [15]:
# Contrastive loss function for bi-encoder training
class ContrastiveLoss(torch.nn.Module):
    def __init__(self, temperature=0.05):
        super().__init__()
        self.temperature = temperature
        self.criterion = torch.nn.CrossEntropyLoss()

    def forward(self, query_embeddings, passage_embeddings):
        query_embeddings = torch.nn.functional.normalize(query_embeddings, p=2, dim=1)
        passage_embeddings = torch.nn.functional.normalize(passage_embeddings, p=2, dim=1)
        # Compute cosine similarities
        similarities = torch.matmul(query_embeddings, passage_embeddings.T) / self.temperature
        labels = torch.arange(similarities.size(0)).long().to(similarities.device)  # Positive pairs on the diagonal
        loss = self.criterion(similarities, labels)
        return loss


### Trainer

In [16]:
class BiEncoderTrainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        query_embeddings, passage_embeddings = model(
            inputs['input_ids_query'], inputs['attention_mask_query'],
            inputs['input_ids_passage'], inputs['attention_mask_passage']
        )
        loss = loss_fn(query_embeddings, passage_embeddings)
        return (loss, (query_embeddings, passage_embeddings)) if return_outputs else loss


NameError: name 'Trainer' is not defined

# Querying

In [ ]:
import asyncio
import numpy as np
from typing import List, Tuple, Any

_TOP_K = 100

async def _fetch_field_hits(doc_index: Any, field: str, query: str, k: int):
    k = min(k, _TOP_K)
    result = await asyncio.to_thread(doc_index.search, field, query, k)
    hits = list(result)
    out = []
    for item in hits:
        if isinstance(item, (tuple, list)) and len(item) >= 2:
            out.append((int(item[0]), float(item[1])))
        else:
            raise ValueError("doc_index.search must yield (doc_id, score) pairs")
    return out

def _compute_cols_scores_sync(hits, docid2col):
    if not hits:
        return np.array([], dtype=np.int64), np.array([], dtype=np.float32)
    ids_arr = np.fromiter((h[0] for h in hits), dtype=np.int64)
    sc_arr = np.fromiter((h[1] for h in hits), dtype=np.float32)
    cols = np.fromiter((docid2col.get(int(x), -1) for x in ids_arr), dtype=np.int64)
    valid = cols >= 0
    return cols[valid], sc_arr[valid]

async def _compute_cols_scores(hits, docid2col):
    return await asyncio.to_thread(_compute_cols_scores_sync, hits, docid2col)

async def aggregate_field_scores_async(query: str, fields: List[str], doc_index: Any, k: int = 100
                                      ) -> Tuple[List[int], np.ndarray]:
    # concurrently fetch top-k per field
    fetch_tasks = [_fetch_field_hits(doc_index, field, query, k) for field in fields]
    per_field_hits = await asyncio.gather(*fetch_tasks)

    # union of doc ids (deterministic sorted order)
    union_set = set()
    for hits in per_field_hits:
        for doc_id, _ in hits:
            union_set.add(doc_id)
    doc_ids = np.array(sorted(union_set), dtype=np.int64)
    if doc_ids.size == 0:
        return [], np.zeros((len(fields), 0), dtype=np.float32)

    docid2col = {int(d): i for i, d in enumerate(doc_ids)}
    scores = np.zeros((len(fields), doc_ids.size), dtype=np.float32)

    # compute cols and scores for each field in parallel (CPU work offloaded to threads)
    compute_tasks = [_compute_cols_scores(per_field_hits[i], docid2col) for i in range(len(fields))]
    results = await asyncio.gather(*compute_tasks)

    # assign rows using numpy advanced indexing (done in main thread)
    for fi, (cols, sc_arr) in enumerate(results):
        if cols.size:
            scores[fi, cols] = sc_arr

    return doc_ids.tolist(), scores

def aggregate_field_scores(query: str, fields: List[str], doc_index: Any, k: int = 100):
    return asyncio.run(aggregate_field_scores_async(query, fields, doc_index, k))

In [18]:
import time
import random
import numpy as np

class FakeDocIndex:
    def __init__(self, seed=42):
        rng = random.Random(seed)
        # create 3 posting lists (200 each) with overlapping docids
        # field A: 0..199, B:100..299, C:200..399
        self.postings = {
            "title":  [(i, rng.random()) for i in range(0, 200)],
            "body":   [(i, rng.random()) for i in range(100, 300)],
            "tags":   [(i, rng.random()) for i in range(200, 400)]
        }
        # sort by score descending so "top-k" is meaningful
        for f in self.postings:
            self.postings[f].sort(key=lambda x: x[1], reverse=True)
        # per-field artificial synchronous delay (seconds)
        self.delays = {"title": 0.12, "body": 0.08, "tags": 0.05}

    def search(self, field: str, query: str, k: int = 100):
        """Synchronous search API expected by aggregate_field_scores (used via to_thread)."""
        # simulate work / I/O
        time.sleep(self.delays.get(field, 0.05))
        # return top-k (doc_id, score)
        lst = self.postings.get(field, [])
        return lst[:k]

# instantiate fake index
doc_index = FakeDocIndex(seed=123)

# fields and query
fields = ["title", "body", "tags"]
query = "dog"
k = 100  # per-field top-k

# time the aggregate function (uses asyncio internally)
t0 = time.time()
doc_ids, scores = aggregate_field_scores(query, fields, doc_index, k=k)
elapsed = time.time() - t0

print(f"elapsed: {elapsed:.3f}s")
print("num union docs:", len(doc_ids))
print("scores.shape:", scores.shape)

# show top-10 combined docs by sum of field scores
combined = scores.sum(axis=0)
order = np.argsort(-combined)  # descending
top_n = min(10, combined.size)
print("top docs (docid, combined_score):")
for idx in order[:top_n]:
    print(int(doc_ids[idx]), float(combined[idx]))

elapsed: 0.124s
num union docs: 247
scores.shape: (3, 247)
top docs (docid, combined_score):
133 1.9785733222961426
145 1.8747471570968628
278 1.8505465984344482
274 1.8207712173461914
105 1.805562973022461
253 1.803251028060913
212 1.763770580291748
229 1.7281708717346191
195 1.6996092796325684
136 1.6826417446136475
